# Projet ML - Classification du Churn Client Telco

**Dataset:** WA_Fn-UseC_-Telco-Customer-Churn.csv (7043 observations)  
**Objectif:** Prédire si un client va quitter l'entreprise (Churn)  
**Modèles:** Régression Logistique, SVM, Random Forest, Gradient Boosting  
**Interprétabilité:** SHAP (global + local), Feature Importance, LIME

Python: 3.10+ | Exécution: `uv run jupyter notebook telco_churn_ml.ipynb`

**Installation des dépendances** — Exécuter la cellule ci-dessous en premier si les paquets ne sont pas encore installés.

In [ ]:
# Installation des paquets si manquants (exécuter cette cellule en premier)
import sys
import subprocess

def ensure_packages():
    required = [
        ("numpy", "numpy"),
        ("pandas", "pandas"),
        ("sklearn", "scikit-learn"),
        ("matplotlib", "matplotlib"),
        ("seaborn", "seaborn"),
        ("shap", "shap"),
        ("lime", "lime"),
    ]
    to_install = []
    for import_name, pip_name in required:
        try:
            __import__(import_name)
        except ImportError:
            to_install.append(pip_name)
    if to_install:
        print(f"Installation des paquets manquants: {to_install}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + to_install)
    else:
        print("Tous les paquets sont déjà installés.")

ensure_packages()

## 1. Imports et configuration

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import shap
from lime import lime_tabular

DATA_PATH = Path("WA_Fn-UseC_-Telco-Customer-Churn.csv")
RANDOM_STATE = 42
TEST_SIZE = 0.2

BEST_MODEL_PARAMS = {
    "n_estimators": 200,
    "max_depth": 12,
    "min_samples_split": 5,
    "min_samples_leaf": 2,
    "max_features": "sqrt",
    "bootstrap": True,
    "random_state": RANDOM_STATE,
}

ModuleNotFoundError: No module named 'numpy'

## 2. Chargement et exploration des données

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
print(f"\nRépartition Churn:\n{df['Churn'].value_counts()}")
df.head()

## 3. Préprocessing

In [ ]:
df = df.drop(columns=["customerID"])
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.dropna(subset=["TotalCharges"])

y = df["Churn"].map({"Yes": 1, "No": 0})
X = df.drop(columns=["Churn"])

for col in X.select_dtypes(include=["object"]).columns:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)
feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
X_train_np = np.array(X_train)
X_test_np = np.array(X_test)
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")

## 4. Entraînement et comparaison des modèles

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, C=0.5),
    "SVM (RBF)": SVC(kernel="rbf", C=1.0, gamma="scale", probability=True),
    "Random Forest": RandomForestClassifier(**BEST_MODEL_PARAMS),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=RANDOM_STATE),
}

for name, model in models.items():
    scores = cross_val_score(model, X_train_np, y_train, cv=5, scoring="f1")
    model.fit(X_train_np, y_train)
    print(f"{name}: F1 (CV) = {scores.mean():.4f} ± {scores.std():.4f}")

## 5. Évaluation sur le set de test

In [ ]:
for name, model in models.items():
    y_pred = model.predict(X_test_np)
    y_proba = model.predict_proba(X_test_np)[:, 1] if hasattr(model, "predict_proba") else None
    print(f"--- {name} ---")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f} | Precision: {precision_score(y_test, y_pred, zero_division=0):.4f} | Recall: {recall_score(y_test, y_pred, zero_division=0):.4f} | F1: {f1_score(y_test, y_pred, zero_division=0):.4f}")
    if y_proba is not None:
        print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")
    print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}\n")

## 6. Interprétabilité globale (Feature Importance + SHAP)

In [ ]:
best_model = models["Random Forest"]
importance = pd.Series(best_model.feature_importances_, index=feature_names).sort_values(ascending=False)
importance.head(10)

In [ ]:
plt.figure(figsize=(10, 6))
importance.head(15).plot(kind="barh")
plt.title("Feature Importance - Random Forest")
plt.tight_layout()
plt.show()

In [ ]:
X_df = pd.DataFrame(X_train_np[:500], columns=feature_names)
explainer = shap.Explainer(best_model, X_df, feature_names=feature_names)
shap_values = explainer(X_df)
sv = shap_values.values[:, :, 1] if len(shap_values.values.shape) == 3 else shap_values.values
shap.summary_plot(sv, X_df, feature_names=feature_names)
plt.show()

## 7. Interprétabilité locale (SHAP Waterfall + LIME)

In [ ]:
shap_vals = explainer(X_df.iloc[:1])
exp = shap_vals[0, :, 1]  # classe Churn
shap.plots.waterfall(exp, max_display=12)
plt.show()

In [ ]:
lime_explainer = lime_tabular.LimeTabularExplainer(
    X_train_np, feature_names=feature_names, class_names=["No Churn", "Churn"], mode="classification"
)
exp = lime_explainer.explain_instance(X_test_np[0], best_model.predict_proba, num_features=10)
print("Observation 1:", exp.as_list())

## 8. Résumé - Hyperparamètres du meilleur modèle

In [ ]:
for k, v in BEST_MODEL_PARAMS.items():
    print(f"  {k}: {v}")